# 08 — Stage 3: Principal Components of the Corrected Residuals

Exploratory check of factor-model completeness: does systematic cross-fund co-movement survive in the Stage 2b
residuals? With N = 1,950 funds and T = 270 months, PCA is done in month space (Connor–Korajczyk asymptotic PCA): the
T×T operator Ω is built from per-fund standardised residuals, pairwise-complete over the unbalanced panel (no
imputation), and its eigenvectors are the factor score series.

| | |
|---|---|
| **Inputs** | `stage2b_residuals.parquet`, `stage2b_results.parquet` (NB06); `panel_with_factors.parquet` (NB03); `analysis_panel.parquet` (NB01) |
| **Outputs** | none (Figures 4–6 saved) |
| **Thesis** | Sections 4.5, 5.5, 5.6, Figures 4–6 |
| **Expected** | 235 months × 1,950 funds; PC1 12.64%, PC2 9.56% of residual variance; λ₂/λ₃ = 2.13 |

**To replicate:** set `DATA_ROOT` in `thesis_config.py`, run NB01–NB06, then *Restart Kernel → Run All*.
The permutation test (200 shuffles) takes a few minutes.

## 0. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from matplotlib.patches import Patch
from scipy import stats

from thesis_config import DATA_DIR, assert_matches, check_paths, save_fig

check_paths(*(DATA_DIR / f for f in ["stage2b_residuals.parquet", "stage2b_results.parquet",
                                     "panel_with_factors.parquet", "analysis_panel.parquet"]))
print(f"pandas {pd.__version__} | numpy {np.__version__}")

C_VALUE, C_GROWTH, C_OTHER = "#1f5fb4", "#c0392b", "#7f8c8d"
C_FULL,  C_CALM            = "#1f5fb4", "#2e8b57"

## 1. Residual matrix and month trimming

In [ ]:
resid = pd.read_parquet(DATA_DIR / "stage2b_residuals.parquet")
assert not resid.duplicated(subset=["wficn", "caldt"]).any()

wide = resid.pivot(index="caldt", columns="wficn", values="residual").sort_index()
T, N = wide.shape
obs = wide.notna()
print(f"T = {T} months × N = {N} funds, {obs.values.mean():.1%} of cells observed, N/T = {N/T:.1f}")
assert_matches("Section 4.5: T, N", [T, N], [270, 1_950], 0)
assert_matches("Section 4.5: N/T", N / T, 7.2, 1)

overlap_all = obs.to_numpy(float) @ obs.to_numpy(float).T
print(f"Untrimmed: minimum fund overlap of a month pair = {overlap_all[~np.eye(T, dtype=bool)].min():.0f}")

funds_per_month = obs.sum(axis=1)
for thr in [50, 100, 150, 200]:
    keep = funds_per_month[funds_per_month >= thr].index
    sub = obs.loc[keep].to_numpy(float)
    ov = sub @ sub.T
    print(f"  ≥ {thr:>3} funds/month: {len(keep)} months ({keep.min():%Y-%m} to {keep.max():%Y-%m}), "
          f"min pairwise overlap {ov[~np.eye(len(ov), dtype=bool)].min():.0f}")

In [ ]:
MIN_FUNDS_PER_MONTH = 100
keep_months = funds_per_month[funds_per_month >= MIN_FUNDS_PER_MONTH].index
wide_t = wide.loc[keep_months]

# Standardise each fund over the trimmed months: PCA on the correlation structure, every fund weighted equally
sd_t   = wide_t.std()
dead_t = sd_t[(sd_t == 0) | sd_t.isna()].index
wide_t = wide_t.drop(columns=dead_t)
Zt = (wide_t - wide_t.mean()) / wide_t.std()

Z0t      = Zt.fillna(0).values
Mt       = Zt.notna().to_numpy(dtype=float)
counts_t = Mt @ Mt.T                                          # funds observed in both months
Omega_t  = (Z0t @ Z0t.T) / np.where(counts_t > 0, counts_t, np.nan)
print(f"Trimmed panel: {Zt.shape[0]} months × {Zt.shape[1]} funds ({keep_months.min():%Y-%m} to {keep_months.max():%Y-%m}); "
      f"min pairwise overlap {counts_t[~np.eye(len(counts_t), dtype=bool)].min():.0f}; dropped funds: {len(dead_t)}")
assert_matches("Section 4.5: trimmed panel (months, funds)", Zt.shape, (235, 1_950), 0)

## 2. Eigenvalues, noise ceiling and number of components

In [ ]:
evals_t, evecs_t = np.linalg.eigh(np.nan_to_num(Omega_t, nan=0.0))
evals_t, evecs_t = evals_t[::-1], evecs_t[:, ::-1]           # descending

neg_idx = np.where(evals_t < 0)[0]
print(f"Negative eigenvalues (pairwise operator is not PSD): {len(neg_idx)}, at positions "
      f"{neg_idx.min()+1}–{neg_idx.max()+1} of {len(evals_t)}; most negative = {evals_t.min():.3f} vs top = {evals_t[0]:.3f}")

pos_sum = evals_t[evals_t > 0].sum()
share = evals_t / pos_sum * 100
print(f"\n{'PC':>3} {'eigenvalue':>11} {'% var':>7} {'cum %':>7}")
for k in range(10):
    print(f"{k+1:>3} {evals_t[k]:>11.4f} {share[k]:>6.2f}% {share[:k+1].sum():>6.2f}%")

**Permutation noise ceiling.** Each fund's residuals are shuffled in time independently (destroying cross-fund
co-movement, keeping each fund's variance and missingness pattern); the operator is rebuilt and its top eigenvalues
recorded, 200 times.

In [ ]:
RNG, N_PERM, K_TRACK = np.random.default_rng(42), 200, 15

Zt_vals = Zt.to_numpy()
mask    = Zt.notna().to_numpy()
cnt     = mask.astype(float) @ mask.astype(float).T

perm_top = np.empty((N_PERM, K_TRACK))
for p in range(N_PERM):
    Zs = Zt_vals.copy()
    for j in range(Zs.shape[1]):
        o = np.where(mask[:, j])[0]
        Zs[o, j] = Zs[RNG.permutation(o), j]
    Zs0  = np.nan_to_num(Zs, nan=0.0)
    Om_s = (Zs0 @ Zs0.T) / np.where(cnt > 0, cnt, np.nan)
    perm_top[p] = np.linalg.eigvalsh(np.nan_to_num(Om_s, nan=0.0))[::-1][:K_TRACK]

null_p95 = np.percentile(perm_top, 95, axis=0)
n_real = int((evals_t[:K_TRACK] > null_p95).sum())
print(f"{'PC':>3} {'actual λ':>9} {'null p95':>9}")
for k in range(K_TRACK):
    print(f"{k+1:>3} {evals_t[k]:>9.3f} {null_p95[k]:>9.3f}")
print(f"\nComponents above the 95th-percentile noise ceiling: {n_real} of {K_TRACK} tracked")

ratios = evals_t[:14] / evals_t[1:15]
print("Eigenvalue ratios λk/λk+1: " + ", ".join(f"{r:.2f}" for r in ratios))
print(f"Largest ratio at k = {np.argmax(ratios)+1} -> retain {np.argmax(ratios)+1} components")

assert_matches("Section 5.5: % variance PC1, PC2; λ2/λ3", [share[0], share[1], ratios[1]], [12.64, 9.56, 2.13], 2)
assert_matches("Section 5.5: components above noise ceiling; retained", [n_real, np.argmax(ratios) + 1], [15, 2], 0)

## 3. Section 5.5 — what are PC1 and PC2?

Fund loading = correlation of the fund's standardised residual with the component's score series over the fund's
observed months.

In [ ]:
scores = pd.DataFrame(evecs_t[:, :3], index=Zt.index, columns=["PC1", "PC2", "PC3"])
load = pd.DataFrame(index=Zt.columns, columns=["PC1", "PC2", "PC3"], dtype=float)
for pc in ["PC1", "PC2", "PC3"]:
    s = scores[pc]
    for wf in Zt.columns:
        z = Zt[wf]
        m = z.notna()
        load.loc[wf, pc] = np.corrcoef(z[m], s[m])[0, 1]

print(load.describe(percentiles=[.05, .25, .5, .75, .95]).round(3).to_string())
same_sign = {}
for pc in ["PC1", "PC2"]:
    v = load[pc].dropna()
    same_sign[pc] = max((v > 0).mean(), (v < 0).mean())
    print(f"{pc}: {same_sign[pc]:.0%} of funds load with the same sign, mean {v.mean():+.3f}, |loading| median {v.abs().median():.3f}")
assert_matches("Section 5.5: same-sign share PC1, PC2", [same_sign["PC1"], same_sign["PC2"]], [0.92, 0.54], 2)
assert_matches("Section 5.5: mean PC1 loading", load["PC1"].mean(), -0.286, 3)

**PC1 and market timing.** Stage 2b residuals are orthogonal to the linear market factor fund by fund, so only a
nonlinear market association can show up: PC1 scores on mkt_rf and mkt_rf² with HAC(6) standard errors.

In [ ]:
fac = (pd.read_parquet(DATA_DIR / "panel_with_factors.parquet")[["caldt", "mkt_rf"]]
         .drop_duplicates("caldt").set_index("caldt").reindex(scores.index))
assert fac["mkt_rf"].notna().all()

d = pd.DataFrame({"pc1": scores["PC1"], "mkt": fac["mkt_rf"]}).dropna()
d["mkt2"] = d["mkt"] ** 2
res = sm.OLS(d["pc1"], sm.add_constant(d[["mkt", "mkt2"]])).fit(cov_type="HAC", cov_kwds={"maxlags": 6})
print(res.summary().tables[1])
print(f"R² = {res.rsquared:.3f}")
assert_matches("Section 5.5: mkt² z", res.tvalues["mkt2"], 1.56, 2)
assert_matches("Section 5.5: mkt² p, R²", [res.pvalues["mkt2"], res.rsquared], [0.119, 0.025], 3)

**PC2 and fund style.** Fund attributes: Lipper objective code, size bucket of the fund's first month (`mtna`, $M).
Style camps for the contrast: growth = {TK, CA, G}, value = {EI, H, GI, FS}; cap- or sector-defined codes are "other".

In [ ]:
panel = pd.read_parquet(DATA_DIR / "analysis_panel.parquet").sort_values(["wficn", "caldt"])
AUM_ORDER = ["$50-100M", "$100-250M", "$250-500M", "$500M-1B", "$1-5B", "$5-10B", ">$10B"]
panel["aum_bucket"] = pd.cut(panel["mtna"], bins=[50, 100, 250, 500, 1000, 5000, 10000, np.inf],
                             labels=AUM_ORDER, right=False, ordered=True)
attrs = (panel.groupby("wficn", as_index=False)
              .agg(lipper_obj_cd=("lipper_obj_cd", "first"), aum_bucket=("aum_bucket", "first")))
attrs["aum_bucket"] = pd.Categorical(attrs["aum_bucket"], categories=AUM_ORDER, ordered=True)

GROWTH, VALUE = {"TK", "CA", "G"}, {"EI", "H", "GI", "FS"}
lj = load.reset_index().rename(columns={"index": "wficn"}).merge(attrs, on="wficn", how="left")
lj["style"] = lj["lipper_obj_cd"].map(lambda c: "growth" if c in GROWTH else ("value" if c in VALUE else "other"))
assert lj["lipper_obj_cd"].notna().all()

by_obj = (lj.groupby("lipper_obj_cd")["PC2"].agg(["mean", "count"]).query("count >= 20").sort_values("mean"))
print("Mean PC2 loading by Lipper objective (categories with ≥ 20 funds):")
print(by_obj.round(3).to_string())

sub = lj[lj["style"].isin(["growth", "value"])].copy()
print(f"\nStyle contrast: {len(sub)} funds (growth {(sub['style'] == 'growth').sum()}, value {(sub['style'] == 'value').sum()})")

**Figure 4 — PC2 loading ordered by Lipper style mandate.**

In [ ]:
colour = lambda c: C_GROWTH if c in GROWTH else (C_VALUE if c in VALUE else C_OTHER)
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(by_obj.index, by_obj["mean"], color=[colour(c) for c in by_obj.index])
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Mean PC2 loading")
ax.set_ylabel("Lipper objective code")
ax.set_title("PC2 loading sorts by style mandate")
ax.legend(handles=[Patch(color=C_GROWTH, label="Growth"), Patch(color=C_VALUE, label="Value"),
                   Patch(color=C_OTHER, label="Other")], loc="lower right")
save_fig(fig, "fig4_pc2_loading_by_lipper_code")
plt.show()

**Size control.** Value-minus-growth gap in mean PC2 loading within each size bucket. The loadings are not independent
across funds, so inference uses a one-sided sign test on the seven bucket gaps rather than a pooled t-statistic
(the pooled Welch *t* is printed for reference only).

In [ ]:
def gap_by_bucket(frame, col):
    t = frame.groupby(["aum_bucket", "style"], observed=True)[col].mean().unstack("style")
    return (t["value"] - t["growth"]).reindex(AUM_ORDER)

tab = sub.groupby(["aum_bucket", "style"], observed=True)["PC2"].mean().unstack("style")
tab["gap (value − growth)"] = tab["value"] - tab["growth"]
tab["n_growth"] = sub[sub["style"] == "growth"].groupby("aum_bucket", observed=True).size()
tab["n_value"]  = sub[sub["style"] == "value"].groupby("aum_bucket", observed=True).size()
print(tab.round(3).to_string())

g_full = gap_by_bucket(sub, "PC2").dropna()
sign_p = stats.binomtest(int((g_full > 0).sum()), len(g_full), p=0.5, alternative="greater").pvalue
welch  = stats.ttest_ind(sub.loc[sub["style"] == "value", "PC2"], sub.loc[sub["style"] == "growth", "PC2"], equal_var=False)
print(f"\nBuckets with a positive gap: {(g_full > 0).sum()}/{len(g_full)} (one-sided sign test p = {sign_p:.4f}); "
      f"mean within-bucket gap {g_full.mean():+.4f}")
print(f"Pooled Welch t = {welch.statistic:.2f} (reference only)")

assert_matches("Section 5.5: buckets positive, style funds (all, growth, value)",
               [(g_full > 0).sum(), len(sub), (sub["style"] == "growth").sum(), (sub["style"] == "value").sum()],
               [7, 1_231, 768, 463], 0)
assert_matches("Section 6.1: sign-test p", sign_p, 0.008, 3)
assert_matches("Section 5.5: mean within-bucket gap", g_full.mean(), 0.434, 3)   # thesis text prints 0.433 (rounding slip)

## 4. Section 5.6 — robustness of PC2

**Self-labelling.** PC2 loading against each fund's realised seven-factor HML beta (Stage 2b). Residuals are
HML-orthogonal within each fund, so an association here is not mechanical.

In [ ]:
betas = pd.read_parquet(DATA_DIR / "stage2b_results.parquet")[["wficn", "beta_hml"]].rename(columns={"beta_hml": "hml_beta"})
chk = lj.merge(betas.dropna().drop_duplicates("wficn"), on="wficn", how="inner")
rho = chk["PC2"].corr(chk["hml_beta"], method="spearman")
rho_camp = {c: chk.loc[chk["style"] == c, "PC2"].corr(chk.loc[chk["style"] == c, "hml_beta"], method="spearman")
            for c in ["growth", "value"]}
print(f"Spearman(PC2 loading, HML beta): all {rho:+.3f} | within growth {rho_camp['growth']:+.3f} | "
      f"within value {rho_camp['value']:+.3f}  (N = {len(chk)})")
assert_matches("Section 5.6: Spearman all / growth / value", [rho, rho_camp["growth"], rho_camp["value"]],
               [0.697, 0.821, 0.500], 3)

**Figure 5 — PC2 loading versus realised HML beta.**

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for camp, col in {"growth": C_GROWTH, "value": C_VALUE, "other": C_OTHER}.items():
    d = chk[chk["style"] == camp]
    ax.scatter(d["hml_beta"], d["PC2"], s=10, alpha=0.5, color=col, label=camp.capitalize())
ax.axhline(0, color="black", lw=0.8)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Realized HML beta (seven-factor)")
ax.set_ylabel("PC2 loading")
ax.set_title(f"PC2 tracks realized value tilt  (Spearman = {rho:+.3f})")
ax.legend(loc="upper left")
save_fig(fig, "fig5_pc2_vs_hml_beta")
plt.show()

**Episodic artefact.** Drop the most turbulent 15% of months (by mkt_rf²), recompute PC2 loadings on the calm months
only, and repeat the value–growth contrast.

In [ ]:
m2   = fac["mkt_rf"].pow(2)
calm = m2.index[m2 <= m2.quantile(0.85)].intersection(scores.index)
s2c  = scores.loc[calm, "PC2"]
print(f"Dropped {len(scores) - len(calm)} turbulent months, kept {len(calm)} calm months; "
      f"PC2 energy in dropped months: {1 - s2c.pow(2).sum() / scores['PC2'].pow(2).sum():.1%}")

Zc = Zt.loc[calm]
pc2_calm = {}
for wf in Zt.columns:
    z = Zc[wf].dropna()
    if len(z) > 24:
        c = np.corrcoef(z.values, s2c.loc[z.index].values)[0, 1]
        pc2_calm[wf] = c if np.isfinite(c) else np.nan
lj["PC2_calm"] = lj["wficn"].map(pc2_calm).astype(float)

sub2 = lj[lj["style"].isin(["growth", "value"])].dropna(subset=["PC2_calm"]).copy()
gap_calm = sub2.loc[sub2["style"] == "value", "PC2_calm"].mean() - sub2.loc[sub2["style"] == "growth", "PC2_calm"].mean()
g_calm = gap_by_bucket(sub2, "PC2_calm")
print(f"Value−growth gap, calm months only: {gap_calm:+.3f} | buckets positive: {(g_calm.dropna() > 0).sum()}/{g_calm.notna().sum()}")

assert_matches("Section 5.6: turbulent, calm months, buckets positive",
               [len(scores) - len(calm), len(calm), (g_calm > 0).sum()], [36, 199, 7], 0)
assert_matches("Section 5.6: calm-only value−growth gap", gap_calm, 0.356, 3)

**Figure 6 — value-minus-growth PC2 gap by size bucket, full sample vs calm months.**

In [ ]:
x, w = np.arange(len(AUM_ORDER)), 0.4
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, gap_by_bucket(sub, "PC2"), w, color=C_FULL, label="Full sample")
ax.bar(x + w/2, g_calm, w, color=C_CALM, label="Calm months only")
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(AUM_ORDER, rotation=30, ha="right")
ax.set_ylabel("Value − growth gap in mean PC2 loading")
ax.set_title("Value–growth gap: positive in every bucket, widening with size")
ax.legend()
save_fig(fig, "fig6_pc2_gap_by_aum_bucket")
plt.show()